# Экстракция без `key` и `description`, с обязательным `source`

Notebook использует отдельную строгую схему и не меняет публичные модели приложения. У каждого параметра обязательны `source_type="user_input"` и дословный непустой `raw_text_fragment`. Для запуска нужен работающий локальный Ollama.

In [1]:
import json
import sys
from pathlib import Path
from typing import Any, Literal

PROJECT_ROOT = next(
    (path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (path / "pyproject.toml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Не найден корень проекта с pyproject.toml")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

MY_TESTS_DIR = PROJECT_ROOT / "my_tests"
RAW_INPUT_PATH = MY_TESTS_DIR / "raw_prompt.txt"
OUTPUT_PATH = MY_TESTS_DIR / "extracted_document_required_source.json"

In [2]:
from pydantic import BaseModel, ConfigDict, Field

from report_system.config import Settings
from report_system.domain import ValueType
from report_system.llm import OllamaProvider


class RequiredUserSource(BaseModel):
    model_config = ConfigDict(extra="forbid")

    source_type: Literal["user_input"]
    raw_text_fragment: str = Field(min_length=1, max_length=200)


class ExtractedParameter(BaseModel):
    model_config = ConfigDict(extra="forbid")

    name: str = Field(min_length=1)
    value: Any
    unit: str | None = None
    value_type: ValueType = ValueType.SCALAR
    source: RequiredUserSource


class ExtractedRecord(BaseModel):
    model_config = ConfigDict(extra="forbid")

    type: str = Field(min_length=1)
    name: str = Field(min_length=1)
    parameters: list[ExtractedParameter] = Field(default_factory=list)


class ExtractedSection(BaseModel):
    model_config = ConfigDict(extra="forbid")

    name: str = Field(min_length=1)
    records: list[ExtractedRecord] = Field(default_factory=list)


class ExtractionResult(BaseModel):
    model_config = ConfigDict(extra="forbid")

    title: str | None = None
    sections: list[ExtractedSection] = Field(default_factory=list)
    conclusion: str | None = None

In [3]:
schema = ExtractionResult.model_json_schema()
parameter_schema = schema["$defs"]["ExtractedParameter"]
source_schema = schema["$defs"]["RequiredUserSource"]

for definition_name in ("ExtractedParameter", "ExtractedRecord", "ExtractedSection"):
    properties = schema["$defs"][definition_name]["properties"]
    assert "key" not in properties
    assert "description" not in properties
assert "source" in parameter_schema["required"]
assert {"source_type", "raw_text_fragment"} <= set(source_schema["required"])

print("Схема корректна: key/description отсутствуют, source обязателен.")

Схема корректна: key/description отсутствуют, source обязателен.


In [4]:
raw_input = RAW_INPUT_PATH.read_text(encoding="utf-8").strip()
if not raw_input:
    raise ValueError(f"Файл {RAW_INPUT_PATH} пуст")

schema_text = json.dumps(schema, ensure_ascii=False)
prompt = f"""
Извлеки только явно указанные факты акта наработки и верни JSON строго по схеме ниже.
Не придумывай факты, названия, значения или единицы. Не меняй числа.
Каждый извлечённый факт оформи как отдельный параметр.
У КАЖДОГО параметра обязательно заполни source:
- source_type всегда равен user_input;
- raw_text_fragment — короткая дословная непрерывная цитата из исходного текста, подтверждающая значение.
Полей key и description в результате быть не должно.

JSON SCHEMA:
{schema_text}

ИСХОДНЫЙ ТЕКСТ:
---
{raw_input}
---
""".strip()

print(f"Прочитано символов: {len(raw_input)}")

Прочитано символов: 692


In [5]:
settings = Settings()
provider = OllamaProvider(
    base_url=settings.ollama_url,
    model=settings.ollama_model,
    timeout=1800.0,
)
response = provider.generate(prompt, json_schema=schema, temperature=0.0)
document = ExtractionResult.model_validate(response)

In [6]:
parameters = [
    parameter
    for section in document.sections
    for record in section.records
    for parameter in record.parameters
]
if not parameters:
    raise AssertionError("Экстрактор не вернул ни одного параметра")

invalid_fragments = [
    parameter.source.raw_text_fragment
    for parameter in parameters
    if parameter.source.raw_text_fragment not in raw_input
]
if invalid_fragments:
    raise AssertionError(f"Найдены недословные source-фрагменты: {invalid_fragments}")

result_json = document.model_dump_json(indent=2)
assert '"key"' not in result_json
assert '"description"' not in result_json
OUTPUT_PATH.write_text(result_json, encoding="utf-8")

print(f"Параметров: {len(parameters)}")
print(f"Source заполнен и проверен: {len(parameters)} из {len(parameters)}")
print(f"Результат: {OUTPUT_PATH.resolve()}")
print(result_json)

Параметров: 26
Source заполнен и проверен: 26 из 26
Результат: /Users/user/Desktop/Code/ReportBot/my_tests/extracted_document_required_source.json
{
  "title": null,
  "sections": [
    {
      "name": "Состав и подготовка",
      "records": [
        {
          "type": "material",
          "name": "Порошок БА-17",
          "parameters": [
            {
              "name": "mass",
              "value": "300",
              "unit": null,
              "value_type": "scalar",
              "source": {
                "source_type": "user_input",
                "raw_text_fragment": "сначала порошок БА-17 — 300 г"
              }
            },
            {
              "name": "solvent_ratio",
              "value": "70/30",
              "unit": null,
              "value_type": "scalar",
              "source": {
                "source_type": "user_input",
                "raw_text_fragment": "Растворитель MEK/EtOH 70/30"
              }
            },
            {
          

## Генерация документа из строгого extraction-JSON

Следующие ячейки читают сохранённый JSON, генерируют значения строго для placeholders `manufacturing_act.docx`, проверяют их и детерминированно заполняют копию шаблона. Поле `description` здесь относится только к placeholder `{{description}}`; в extraction-JSON его по-прежнему нет.

In [7]:
import re

from docx import Document as DocxDocument
from pydantic import BaseModel, ConfigDict, Field

from report_system.config import Settings
from report_system.domain import Document, DocumentStatus, DocumentType, Section
from report_system.llm import OllamaProvider
from report_system.validation import validate_facts

REPORT_TEXT_PATH = MY_TESTS_DIR / "generated_report_required_source.txt"
REPORT_DOCX_PATH = MY_TESTS_DIR / "generated_manufacturing_act_required_source.docx"
TEMPLATE_FIELDS_PATH = MY_TESTS_DIR / "generated_template_fields_required_source.json"

raw_input = RAW_INPUT_PATH.read_text(encoding="utf-8").strip()
settings = Settings()
provider = OllamaProvider(
    base_url=settings.ollama_url,
    model=settings.ollama_model,
    timeout=1800.0,
)

strict_json_text = OUTPUT_PATH.read_text(encoding="utf-8")
assert '"key"' not in strict_json_text
assert '"description"' not in strict_json_text
strict_document = ExtractionResult.model_validate_json(strict_json_text)
strict_parameters = [
    parameter
    for section in strict_document.sections
    for record in section.records
    for parameter in record.parameters
]

report_document = Document(
    id="manual-required-source",
    document_type=DocumentType.MANUFACTURING_ACT,
    title=strict_document.title,
    sections=[
        Section.model_validate(section.model_dump(mode="json"))
        for section in strict_document.sections
    ],
    conclusion=strict_document.conclusion,
    raw_input=raw_input,
    status=DocumentStatus.CONFIRMED,
)

TEMPLATE_PATH = settings.templates_dir / "manufacturing_act.docx"
template_document = DocxDocument(TEMPLATE_PATH)
template_text = "\n".join(paragraph.text for paragraph in template_document.paragraphs)
template_placeholders = set(re.findall(r"\{\{([a-z_]+)\}\}", template_text))
expected_placeholders = {
    "document_id", "date", "title", "code",
    "sample", "goal", "description", "conclusion",
}
if template_placeholders != expected_placeholders:
    raise AssertionError(
        f"Набор placeholders шаблона изменился: {sorted(template_placeholders)}"
    )


class ManufacturingActTemplateFields(BaseModel):
    model_config = ConfigDict(extra="forbid")

    date: str | None = Field(description="Дата или период изготовления; null, если отсутствует в JSON")
    title: str | None = Field(description="Наименование образца; null, если отсутствует в JSON")
    code: str | None = Field(description="Код или лабораторный номер образца; null, если отсутствует в JSON")
    sample: str | None = Field(description="Внешний вид и состояние образца только из подтверждённых фактов")
    goal: str | None = Field(description="Цель работ; null, если она явно не указана")
    description: str = Field(min_length=1, description="Связное краткое описание всех выполненных работ и результатов")
    conclusion: str | None = Field(description="Заключение; null, если оно явно не сформулировано")


template_schema = ManufacturingActTemplateFields.model_json_schema()
template_schema_text = json.dumps(template_schema, ensure_ascii=False)
generation_prompt = f"""
Заполни поля акта наработки строго по JSON Schema.
Используй только факты из подтверждённого extraction-JSON. Не используй исходный текст, знания модели или текст самого шаблона как источник фактов.
Не придумывай даты, коды, цели, заключения, действия, материалы, значения и единицы. Не меняй числа.
Если для date, title, code, sample, goal или conclusion нет подтверждённого факта, верни null. Не пиши строку "Не указано": её добавит renderer.
description должна быть связным техническим описанием подтверждённых операций и результатов без заголовков, подписей и повторения полей шаблона.
document_id не генерируй: он заполняется программно.

JSON SCHEMA ПОЛЕЙ ШАБЛОНА:
{template_schema_text}

ТЕКСТ ШАБЛОНА (только структура, не источник фактов):
---
{template_text}
---

ПОДТВЕРЖДЁННЫЙ EXTRACTION-JSON:
{strict_json_text}
""".strip()

print(f"JSON для генерации: {OUTPUT_PATH.resolve()}")
print(f"Шаблон: {TEMPLATE_PATH.resolve()}")
print(f"Placeholders: {sorted(template_placeholders)}")
print(f"Секций: {len(strict_document.sections)}; параметров: {len(strict_parameters)}")

JSON для генерации: /Users/user/Desktop/Code/ReportBot/my_tests/extracted_document_required_source.json
Шаблон: /Users/user/Desktop/Code/ReportBot/templates/manufacturing_act.docx
Placeholders: ['code', 'conclusion', 'date', 'description', 'document_id', 'goal', 'sample', 'title']
Секций: 1; параметров: 26


In [8]:
generated_response = provider.generate(
    generation_prompt,
    json_schema=template_schema,
    temperature=0.0,
)
if not isinstance(generated_response, dict):
    raise TypeError("Генератор полей шаблона должен вернуть JSON-объект")

template_content = ManufacturingActTemplateFields.model_validate(generated_response)
template_content_json = template_content.model_dump_json(indent=2)
TEMPLATE_FIELDS_PATH.write_text(template_content_json, encoding="utf-8")

print(f"Поля шаблона: {TEMPLATE_FIELDS_PATH.resolve()}")
print(template_content_json)

Поля шаблона: /Users/user/Desktop/Code/ReportBot/my_tests/generated_template_fields_required_source.json
{
  "date": null,
  "title": null,
  "code": null,
  "sample": "5 листов; один край у листа №2 с утолщением; толщина сухой ленты 0,31-0,34 мм",
  "goal": null,
  "description": "Подготовлен порошок БА-17 (300 г) с растворителем MEK/EtOH (70/30, всего 190 г), диспергатором BYK-111 (3,0 г) и связкой PMMA (24 г). Смесь перемешивали 3 ч при вязкости 3,8 Па·с (температура 25 °C). Добавлено 8 г растворителя, пластификатор DBP (12 г), смесь перемешана еще 16 ч на роллах. Вязкость снизилась до 3,1 Па·с. Состав отфильтрован через сетку 100 мкм после вакуумирования при -0,08 МПа в течение примерно 15 мин. Полученный состав нанесен ножом с зазором 0,75 мм на силиконизированный ПЭТ со скоростью каретки 0,6 м/мин. Образцы сушились ночь при температуре 21-23 °C. Итоговая годная площадь составила 1,7 м².",
  "conclusion": null
}


In [9]:
generated_field_values = [
    value
    for value in template_content.model_dump().values()
    if value is not None
]
fact_validation = validate_facts(report_document, "\n".join(generated_field_values))
if not fact_validation.valid:
    raise AssertionError(
        "Поля шаблона содержат неподтверждённые числа, даты или идентификаторы:\n"
        + fact_validation.model_dump_json(indent=2)
    )

template_values = {
    "document_id": report_document.id,
    **{
        name: value if value is not None else "Не указано"
        for name, value in template_content.model_dump().items()
    },
}
if set(template_values) != expected_placeholders:
    raise AssertionError("Не для всех placeholders подготовлены значения")

filled_document = DocxDocument(TEMPLATE_PATH)
for paragraph in filled_document.paragraphs:
    for field_name, value in template_values.items():
        placeholder = "{{" + field_name + "}}"
        for run in paragraph.runs:
            if placeholder in run.text:
                run.text = run.text.replace(placeholder, value)

generated_text = "\n".join(paragraph.text for paragraph in filled_document.paragraphs)
unresolved_placeholders = re.findall(r"\{\{[^{}]+\}\}", generated_text)
if unresolved_placeholders:
    raise AssertionError(f"Не заменены placeholders: {unresolved_placeholders}")

required_template_lines = (
    "Акт наработки №",
    "1.\tНаименование образцов:",
    "2.\tКод (шифр, лабораторный номер) образца:",
    "3.\tДата изготовления (или период):",
    "4.\tВнешний вид образца:",
    "5.\tМесто наработки образцов:",
    "6.\tЦель работ:",
    "7.\tКраткое описание работ:",
    "8.\tЗаключение:",
)
missing_lines = [line for line in required_template_lines if line not in generated_text]
if missing_lines:
    raise AssertionError(f"Нарушена структура шаблона: {missing_lines}")

REPORT_TEXT_PATH.write_text(generated_text, encoding="utf-8")
filled_document.save(REPORT_DOCX_PATH)
if not REPORT_DOCX_PATH.exists():
    raise AssertionError(f"DOCX не создан: {REPORT_DOCX_PATH}")

report_document.title = template_content.title
report_document.conclusion = template_content.conclusion
report_document.generated_text = generated_text
report_document.status = DocumentStatus.GENERATED
if report_document.status != DocumentStatus.GENERATED:
    raise AssertionError("Документ не переведён в статус generated")

print("Проверка фактов и структуры шаблона пройдена.")
print(f"Текст: {REPORT_TEXT_PATH.resolve()}")
print(f"DOCX: {REPORT_DOCX_PATH.resolve()}")
print(f"Статус: {report_document.status.value}")
print("\n" + generated_text)

Проверка фактов и структуры шаблона пройдена.
Текст: /Users/user/Desktop/Code/ReportBot/my_tests/generated_report_required_source.txt
DOCX: /Users/user/Desktop/Code/ReportBot/my_tests/generated_manufacturing_act_required_source.docx
Статус: generated

Федеральное государственное автономное образовательное учреждение высшего образования «Московский физико-технический институт (национальный исследовательский университет)»
Акт наработки №manual-required-source
От Не указано
 
1.	Наименование образцов: Не указано 
2.	Код (шифр, лабораторный номер) образца: Не указано
3.	Дата изготовления (или период): Не указано
4.	Внешний вид образца: 5 листов; один край у листа №2 с утолщением; толщина сухой ленты 0,31-0,34 мм
5.	Место наработки образцов: МФТИ, Физтех, г. Долгопрудный, ул. Научный переулок, д.4, лаборатория пост-литий-ионных электрохимических систем 
6.	Цель работ: Не указано
7.	Краткое описание работ: Подготовлен порошок БА-17 (300 г) с растворителем MEK/EtOH (70/30, всего 190 г), диспе